# 06 · MCP Server Walkthrough — Talking to Claude

The same tools we used directly in notebooks 01-04 are exposed over
the Model Context Protocol via FastMCP. This notebook shows:

1. How to launch the server.
2. The tool catalog as the MCP client sees it.
3. A round-trip invocation over stdio.
4. The resource + prompt URIs the server publishes.

This notebook **does not** spawn the server inside Jupyter (FastMCP's
stdio transport claims the kernel's stdio). Run the server in a
separate terminal, then connect to it.

## 1 · Launch — pick a transport

```bash
# Option A — stdio (recommended for Claude Desktop / local agents)
python scripts/run_mcp_stdio.py

# Option B — HTTP/SSE (recommended for browser / remote agents)
python scripts/run_mcp_http.py --host 127.0.0.1 --port 8765
```

For Claude Desktop, drop something like this into your
`~/Library/Application Support/Claude/claude_desktop_config.json`
(mac) or the Windows equivalent:

```json
{
  "mcpServers": {
    "metaopticsai": {
      "command": "python",
      "args": ["D:/metacode/scripts/run_mcp_stdio.py"]
    }
  }
}
```

## 2 · The tool catalog the server exposes

We can preview exactly what an MCP client will receive without
actually spawning the server — by introspecting the same registry the
adapter consumes.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from metaopticsai.orchestration.controller import MetaOpticsController
controller = MetaOpticsController.build()

print("MCP tools the server will advertise:\n")
for name in sorted(controller.tools.names()):
    t = controller.tools.get(name)
    print(f"  {name}")
    print(f"     {t.description[:100]}{'...' if len(t.description) > 100 else ''}")

## 3 · Resources

Resources are LLM-readable knowledge artifacts — addressable by URI.
The default server publishes one per file in `rag/corpus/`.

In [ ]:
from metaopticsai.rag.loader import load_corpus
corpus_dir = Path(ROOT) / "src" / "metaopticsai" / "rag" / "corpus"
corpus = load_corpus(corpus_dir)
for stem in sorted(corpus):
    uri = f"photonics://kb/{stem}"
    print(f"  {uri}  ({len(corpus[stem])} chars)")

## 4 · A sample stdio round-trip

Here's what an MCP `tools/call` looks like on the wire. The MCP client
sends a JSON-RPC request; the server returns a result.

Request (client → server):
```json
{
  "jsonrpc": "2.0",
  "id": 7,
  "method": "tools/call",
  "params": {
    "name": "list_materials",
    "arguments": {}
  }
}
```

Response (server → client):
```json
{
  "jsonrpc": "2.0",
  "id": 7,
  "result": {
    "content": [
      {"type": "text", "text": "{\"count\": 14, \"materials\": [...]}"}
    ]
  }
}
```

FastMCP handles all of this — your only job is to write the
`BaseTool` subclass.

## 5 · Sub-server topology (optional)

For larger deployments where you want to scope which tools each agent
sees, the project ships 4 split-domain servers:

- `mcp_servers.rcwa_server`         — sweep + materials + FDTD library
- `mcp_servers.optimization_server` — analysis + optimization + jobs
- `mcp_servers.materials_server`    — materials lookup only
- `mcp_servers.fabrication_server`  — phase + GDS export

Each is a thin shell over the same `tools/` registry; the `subset.py`
helper picks which tools to expose. Start one with:

```bash
python -m metaopticsai.mcp_servers.rcwa_server.server --transport stdio
```

## What you now have

A consistent tool surface that works identically:
- in notebooks (this file),
- in pytest,
- over an MCP-stdio connection to Claude Desktop,
- inside LangGraph workflow nodes,
- over HTTP/SSE for remote agents.

That uniformity is the architectural payoff of the `BaseTool` → adapter
layer — and the reason the same 15 tools can power both a local
notebook session and a remote LLM-driven design loop.